# What Do Reported Autonomous-Driving Crashes Look Like?

This notebook is a fast visual tour of NHTSA's **reported** crashes involving Automated Driving Systems (ADS), Level 2 Advanced Driver Assistance Systems (ADAS), and the small Other/Unknown bucket in the current third-amended Standing General Order (SGO) data.

> **Interpretation guardrail:** counts below are counts of NHTSA reports, **not crash rates or safety rankings**. Reporting criteria, telemetry/awareness, vehicle fleets, mileage/exposure, operating domains, and update/duplicate behavior differ across reporting entities. The data do not provide the exposure denominator needed for manufacturer comparisons.

The dataset keeps one latest available Report Version per NHTSA Report ID. It does **not** merge rows merely because they share a Same Incident ID.

In [ ]:
from pathlib import Path
from collections import Counter
import re

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', 30)

candidates = [
    Path('/kaggle/input/nhtsa-autonomous-driving-crashes/data.csv'),
    Path('release/data.csv'),
    Path('../release/data.csv'),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Could not find data.csv')

df = pd.read_csv(DATA_PATH, low_memory=False)
df['incident_month_dt'] = pd.to_datetime(df['incident_month'], format='%Y-%m', errors='coerce')
print(f'Loaded {len(df):,} latest-report rows and {df.shape[1]-1:,} release columns from {DATA_PATH}')

## 1. Coverage and report structure

The current reporting regime starts June 16, 2025, but incident months can be older when a later/current-regime report version describes an earlier incident. We therefore keep `source_regime` explicit instead of inferring regime from `incident_month`.

In [ ]:
same_counts = df.loc[df['same_incident_id'].fillna('').astype(str).str.strip().ne(''), 'same_incident_id'].value_counts()
summary = pd.Series({
    'latest Report IDs': df['report_id'].nunique(),
    'incident month min': df['incident_month_dt'].min().strftime('%Y-%m'),
    'incident month max': df['incident_month_dt'].max().strftime('%Y-%m'),
    'max selected Report Version': pd.to_numeric(df['report_version'], errors='coerce').max(),
    'rows with >1 current-snapshot version observed': (pd.to_numeric(df['versions_observed_in_current_snapshot']) > 1).sum(),
    'Same Incident IDs linking >1 output Report ID': (same_counts > 1).sum(),
})
display(summary.to_frame('value'))
display(df['source_bucket'].value_counts().rename_axis('source_bucket').to_frame('latest reports'))

## 2. Reported incidents over time

This chart shows the month associated with each selected latest report. A changing line can reflect reporting scope, fleet activity, reporting behavior, source updates, or exposure—not only changes in crash risk.

In [ ]:
monthly = (df.dropna(subset=['incident_month_dt'])
             .groupby(['incident_month_dt', 'source_bucket'])
             .size()
             .unstack(fill_value=0)
             .sort_index())
ax = monthly.plot(figsize=(12, 5), marker='o', linewidth=1.5)
ax.set_title('Latest NHTSA reports by incident month and source bucket')
ax.set_xlabel('Incident month')
ax.set_ylabel('Number of latest reports')
ax.legend(title='NHTSA source bucket')
plt.tight_layout()
plt.show()

## 3. Reporting entities and vehicles

These are simple **report-count distributions**. They are useful for understanding who and what is represented in the file, but they are not comparable safety rates because fleet size, miles driven, operating domain, crash-detection capability, and reportability differ.

In [ ]:
top_entities = df['reporting_entity'].value_counts().head(12).sort_values()
ax = top_entities.plot(kind='barh', figsize=(10, 6))
ax.set_title('Top reporting entities by number of latest reports — not a safety ranking')
ax.set_xlabel('Latest NHTSA reports')
ax.set_ylabel('Reporting entity')
plt.tight_layout()
plt.show()

In [ ]:
top_makes = df['make'].fillna('Unknown').replace('', 'Unknown').value_counts().head(12).sort_values()
ax = top_makes.plot(kind='barh', figsize=(10, 6))
ax.set_title('Vehicle makes appearing most often in latest reports — not a crash rate')
ax.set_xlabel('Latest NHTSA reports')
ax.set_ylabel('Vehicle make')
plt.tight_layout()
plt.show()

## 4. Roadway, weather, and crash characteristics

In [ ]:
roadway = df['roadway_type'].fillna('Unknown').replace('', 'Unknown').value_counts().head(10).sort_values()
ax = roadway.plot(kind='barh', figsize=(9, 5))
ax.set_title('Reported roadway type')
ax.set_xlabel('Latest reports')
plt.tight_layout()
plt.show()

In [ ]:
weather_cols = {
    'weather_clear': 'Clear',
    'weather_cloudy': 'Cloudy',
    'weather_partly_cloudy': 'Partly cloudy',
    'weather_rain': 'Rain',
    'weather_snow': 'Snow',
    'weather_fog_smoke_haze': 'Fog / smoke / haze',
    'weather_severe_wind': 'Severe wind',
    'weather_unk_see_narrative': 'Unknown / see narrative',
}
weather = pd.Series({label: df[col].fillna('').astype(str).str.strip().eq('Y').sum() for col, label in weather_cols.items()}).sort_values()
ax = weather.plot(kind='barh', figsize=(9, 5))
ax.set_title('Weather flags selected in latest reports')
ax.set_xlabel('Latest reports with flag = Y')
plt.tight_layout()
plt.show()

In [ ]:
crash_with = df['crash_with'].fillna('Unknown').replace('', 'Unknown').value_counts().head(12).sort_values()
ax = crash_with.plot(kind='barh', figsize=(9, 6))
ax.set_title('What the reported crash was with')
ax.set_xlabel('Latest reports')
plt.tight_layout()
plt.show()

## 5. Reported injury severity

In [ ]:
severity = df['highest_injury_severity_alleged'].fillna('Unknown').replace('', 'Unknown').value_counts().sort_values()
ax = severity.plot(kind='barh', figsize=(10, 6))
ax.set_title('Highest injury severity alleged in latest reports')
ax.set_xlabel('Latest reports')
plt.tight_layout()
plt.show()

display(severity.sort_values(ascending=False).to_frame('latest reports'))

## 6. Narrative-text exploration

NHTSA's public narratives are highly useful for qualitative/text analysis. We keep the published text and its redactions. Rather than reproducing individual incident narratives here, the exploration below looks at text length and frequent words across the corpus.

In [ ]:
narr = df['narrative'].fillna('').astype(str).str.strip()
lengths = narr[narr.ne('')].str.len()
print(f'Nonblank narratives: {len(lengths):,} / {len(df):,} ({len(lengths)/len(df):.1%})')
display(lengths.describe(percentiles=[.25, .5, .75, .9, .95]).to_frame('characters'))

ax = lengths.clip(upper=lengths.quantile(.99)).plot(kind='hist', bins=40, figsize=(10, 5))
ax.set_title('Narrative length distribution (x-axis clipped at 99th percentile)')
ax.set_xlabel('Characters')
plt.tight_layout()
plt.show()

In [ ]:
stop = set('''the a an and or but if then of to in on at for from by with without as is are was were be been being this that these those it its vehicle vehicles crash incident report reported reporting driver road roadway system automation ads adas nhtsa unknown see narrative subject other into after before during about approximately'''.split())
tokens = []
for text in narr[narr.ne('')]:
    words = re.findall(r"[A-Za-z][A-Za-z'-]{2,}", text.lower())
    tokens.extend(w for w in words if w not in stop)

common = pd.Series(dict(Counter(tokens).most_common(25))).sort_values()
ax = common.plot(kind='barh', figsize=(10, 8))
ax.set_title('Frequent words in public incident narratives')
ax.set_xlabel('Token count')
plt.tight_layout()
plt.show()

## What this dataset cannot tell us

- It does not contain a consistent denominator such as miles driven, vehicle population, or operating-domain exposure.
- Reporting entities have different data access and crash-awareness capabilities.
- ADS and Level 2 ADAS reporting criteria differ.
- A real-world crash can produce multiple reports; `same_incident_id` is useful linkage evidence but is not perfect and is not used here to silently deduplicate.
- Reports can be revised. This dataset selects the highest currently available `report_version` for each `report_id`.
- The third-amended SGO changed reporting scope/fields beginning June 16, 2025. This V1 intentionally avoids forcing the older schema into the current one.

Use these data to describe **reported crash records and their characteristics**, not to infer relative manufacturer safety without appropriate external exposure and study design.